# CardioSafe AI - Notebook 07c: evidence roles and a robust drug-name resolver

This notebook replaces `cardiosafe_kb.py` on your Drive with **version 2**. It fixes what your 07/07b results showed.

**What your diagnostics showed**
- *Bradycardia flagged "strong" for azithromycin, ciprofloxacin, citalopram:* the sentences are **risk-factor statements** ("QT prolongation risk in patients with bradyarrhythmias", "avoid in patients with bradycardia and long QT syndrome"), not "this drug causes bradycardia". The tier described *where in the label* a sentence sits, not *what it says*.
- *Digoxin QT "moderate":* an **interaction** sentence about *other* drugs (dofetilide, sotalol), not digoxin's own effect.
- *Atenolol AV block "strong":* a **contraindication** in patients with heart block. That is exactly the ECG-finding-to-drug relation we want, but it is a different relation from "the drug causes heart block".
- *Chlorpromazine:* no QT or torsade sentence in **any field of any of its 10 labels**. So this is a genuine gap in the FDA label text, not a rule bug. "No statement found" must never be read as "safe".
- *Zofran / Ecosprin:* RxNorm's best-scoring candidate was a concept with no properties (retired or inactive), and brand lookups for Indian brands failed.

**What version 2 does**
1. Every label sentence gets a **role**: `drug_effect` (drug may cause the finding), `patient_condition` (caution or contraindication in patients who have the finding), `interaction` (about co-medication). Evidence is reported per role, with how many of the retrieved labels contain it. The roles are rule-based, so the sentence is always printed next to its role.
2. A **drug-name resolver**: hand-built brand table, then typo-tolerant brand match, then RxNorm (active concepts only, each candidate validated, plus the `getDrugs` route, plus a name-similarity guard so "Dolo 650" can never become "No Dolo"). Anything not trusted is reported as **unrecognised**, which leads to "insufficient evidence" and never a guess.
3. One synonym table (paracetamol = acetaminophen, ...) so FDA labels, DDInter and RxNorm agree on names.

**Please paste the raw printed text** of cells 5, 6, 7 and 9 (select the output and copy it).
Runtime: CPU, about 5 to 10 minutes (about 60 label requests plus about 60 RxNorm requests).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys
WORK = '/content/drive/MyDrive/CardioSafe'; KB = WORK + '/kb'
for d in ('', '/ddinter', '/crediblemeds'): os.makedirs(KB + d, exist_ok=True)
sys.path.insert(0, WORK)
print('DDInter files:', os.listdir(KB + '/ddinter'), '| CredibleMeds files:', os.listdir(KB + '/crediblemeds'))

In [ ]:
%%writefile /content/drive/MyDrive/CardioSafe/cardiosafe_kb.py
"""CardioSafe drug knowledge base.

- RxNorm: free-text drug name -> RxCUI -> ingredient names (https://rxnav.nlm.nih.gov/REST)
- openFDA drug labels: retrieves label sentences about QT prolongation / bradycardia / AV block, with section and citation
- DDInter: drug-drug interaction severity from the user's downloaded CSV files (CC BY-NC-SA 4.0, non-commercial use)
- CredibleMeds (optional): the user's own downloaded list, used locally for evaluation only; never redistributed

Nothing here invents medical claims: every flag carries the label sentence or table row it came from.
"""
import os, re, json, time, glob
import numpy as np
import pandas as pd

RXNAV = 'https://rxnav.nlm.nih.gov/REST'
FDA_LABEL = 'https://api.fda.gov/drug/label.json'
UA = {'User-Agent': 'CardioSafe-AI-course-project'}
STATS = {'requests': 0, 'errors': 0}


# ------------------------------------------------------------------ cache + http
class JsonCache:
    def __init__(self, path):
        self.path, self.d, self.dirty = path, {}, 0
        if path and os.path.exists(path):
            try:
                self.d = json.load(open(path))
            except Exception:
                self.d = {}

    def has(self, k): return k in self.d
    def get(self, k): return self.d.get(k)

    def set(self, k, v):
        self.d[k] = v
        self.dirty += 1
        if self.dirty % 20 == 0:
            self.save()

    def save(self):
        if self.path:
            tmp = self.path + '.tmp'
            json.dump(self.d, open(tmp, 'w'))
            os.replace(tmp, self.path)
        self.dirty = 0


def http_get(url, params=None, retries=3, pause=0.3, timeout=30):
    """GET -> parsed JSON; {'_status': 404/400} for those codes; None on network failure."""
    import requests
    for a in range(retries):
        try:
            r = requests.get(url, params=params, headers=UA, timeout=timeout)
            STATS['requests'] += 1
            if r.status_code in (400, 404):
                return {'_status': r.status_code}
            if r.status_code == 429:
                time.sleep(3 * (a + 1)); continue
            if r.status_code >= 400:
                STATS['errors'] += 1; time.sleep(1 + a); continue
            time.sleep(pause)
            return r.json()
        except Exception:
            STATS['errors'] += 1
            time.sleep(1 + a)
    return None


# ------------------------------------------------------------------- names
SALTS = (r'\b(?:hydrochloride|hcl|sodium|potassium|calcium|sulfate|sulphate|mesylate|maleate|tartrate|besylate|succinate|'
         r'fumarate|phosphate|acetate|citrate|bromide|nitrate|dihydrate|monohydrate|hydrobromide|magnesium|disodium|axetil)\b')


# International non-proprietary names vs the US names used by RxNorm / FDA labels / DDInter, so all sources agree on one name
SYNONYMS = {'paracetamol': 'acetaminophen', 'salbutamol': 'albuterol', 'adrenaline': 'epinephrine', 'noradrenaline': 'norepinephrine',
            'frusemide': 'furosemide', 'glibenclamide': 'glyburide', 'rifampicin': 'rifampin', 'lignocaine': 'lidocaine',
            'acetylsalicylic acid': 'aspirin', 'amoxycillin': 'amoxicillin', 'clavulanic acid': 'clavulanate',
            'clavulanate potassium': 'clavulanate', 'thyroxine': 'levothyroxine'}


def canon(name):
    """Canonical lower-case ingredient name: no salts, parentheses or punctuation; INN synonyms mapped to the US name."""
    s = str(name).lower()
    s = re.sub(r'\(.*?\)', ' ', s)
    s = re.sub(SALTS, ' ', s)
    s = re.sub(r'[^a-z0-9+/\- ]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return SYNONYMS.get(s, s)


# ------------------------------------------------------------------- RxNorm
def rx_approx(term, cache, n=5, option=0):
    """RxNorm approximate match. option=1 restricts to active concepts (retired concepts have no properties)."""
    key = f'approx|{option}|{term.lower().strip()}'
    if cache.has(key):
        return cache.get(key)
    js = http_get(f'{RXNAV}/approximateTerm.json', {'term': term, 'maxEntries': n, 'option': option})
    if js is None:
        return None
    out = []
    for c in (js.get('approximateGroup') or {}).get('candidate', []) or []:
        out.append({'rxcui': c.get('rxcui'), 'score': float(c.get('score') or 0), 'rank': int(c.get('rank') or 0),
                    'name': c.get('name'), 'source': c.get('source')})
    cache.set(key, out)
    return out


def rx_drugs(term, cache):
    """RxNorm getDrugs: clinical/branded drug concepts for a name (works well for brand names)."""
    key = f'drugs|{term.lower().strip()}'
    if cache.has(key):
        return cache.get(key)
    js = http_get(f'{RXNAV}/drugs.json', {'name': term})
    if js is None:
        return None
    out = []
    for g in (js.get('drugGroup') or {}).get('conceptGroup', []) or []:
        for c in g.get('conceptProperties', []) or []:
            out.append({'rxcui': c.get('rxcui'), 'name': c.get('name'), 'tty': c.get('tty')})
    out = out[:8]
    cache.set(key, out)
    return out


def rx_props(rxcui, cache):
    key = f'props|{rxcui}'
    if cache.has(key):
        return cache.get(key)
    js = http_get(f'{RXNAV}/rxcui/{rxcui}/properties.json')
    if js is None:
        return None
    p = js.get('properties') or {}
    out = {'rxcui': rxcui, 'name': p.get('name'), 'tty': p.get('tty')}
    cache.set(key, out)
    return out


def rx_ingredients(rxcui, cache):
    """Ingredient names of any RxNorm concept (brand, clinical drug, salt form, ingredient). [] if it cannot be resolved."""
    key = f'ing|{rxcui}'
    if cache.has(key):
        return cache.get(key)
    props = rx_props(rxcui, cache)
    if props is None:
        return None
    if props.get('tty') == 'IN' and props.get('name'):
        names = [props['name']]
    else:
        js = http_get(f'{RXNAV}/rxcui/{rxcui}/related.json', {'tty': 'IN'})
        if js is None:
            return None
        names = []
        for g in (js.get('relatedGroup') or {}).get('conceptGroup', []) or []:
            if g.get('tty') == 'IN':
                names += [c['name'] for c in g.get('conceptProperties', []) or [] if c.get('name')]
    cache.set(key, names)
    return names


def _clean_query(text):
    """Free-text prescription line -> bare drug name: strengths, units and dosage-form words removed."""
    t = re.sub(r'\b\d+(?:[.,]\d+)?\s*(?:mg|mcg|ug|g|ml|iu|units?|%)?\b', ' ', text.lower())
    t = re.sub(r'\b(?:tablets?|tabs?|capsules?|caps?|syrup|injection|inj|oral|er|sr|xl|xr|cr|mr|od|bd|tds)\b', ' ', t)
    return ' '.join(re.sub(r'[^a-z ]', ' ', t).split())


def name_similarity(a, b):
    """0-100 string similarity between two drug names (rapidfuzz if installed, else difflib)."""
    a, b = (a or '').lower().strip(), (b or '').lower().strip()
    if not a or not b:
        return 0.0
    try:
        from rapidfuzz import fuzz
        return float(fuzz.ratio(a, b))
    except Exception:
        import difflib
        return 100.0 * difflib.SequenceMatcher(None, a, b).ratio()


def _brand_in_brackets(name):
    m = re.search(r'\[(.+?)\]', name or '')
    return m.group(1) if m else None


def rx_normalize(text, cache, min_similarity=85):
    """Free text -> ingredient list with an honest status.
    status: 'ok' (resolved AND the query resembles the matched name), 'low_similarity' (resolved but the match looks
    unrelated: treat as unrecognised), 'no_ingredient', 'no_candidate'. Routes tried: approximate match on active concepts,
    approximate match on all concepts, getDrugs. Several candidates are tried, not just the best-scored one."""
    q = _clean_query(text) or text.lower()
    best = None

    def consider(rxcui, names, route, score=None):
        nonlocal best
        ings = rx_ingredients(rxcui, cache) or []
        if not ings:
            return False
        sims = [name_similarity(q, n) for n in names if n] + [name_similarity(q, i) for i in ings]
        sim = max(sims) if sims else 0.0
        cand = {'query': text, 'rxcui': rxcui, 'matched': next((n for n in names if n), None), 'route': route, 'score': score,
                'similarity': round(sim, 1), 'ingredients': sorted({canon(i) for i in ings}),
                'status': 'ok' if sim >= min_similarity else 'low_similarity'}
        if best is None or (cand['status'] == 'ok' and best['status'] != 'ok') or (cand['status'] == best['status'] and cand['similarity'] > best['similarity']):
            best = cand
        return cand['status'] == 'ok'

    seen = set()
    for option in (1, 0):
        for c in sorted(rx_approx(q, cache, 5, option) or [], key=lambda c: -c['score']):
            if not c['rxcui'] or c['rxcui'] in seen:
                continue
            seen.add(c['rxcui'])
            props = rx_props(c['rxcui'], cache) or {}
            if consider(c['rxcui'], [c.get('name'), props.get('name')], f'approx_opt{option}', c['score']):
                return best
    for c in rx_drugs(q, cache) or []:
        if c['rxcui'] and consider(c['rxcui'], [c.get('name'), _brand_in_brackets(c.get('name'))], 'getDrugs'):
            return best
    if best:
        return best
    return {'query': text, 'rxcui': None, 'matched': None, 'route': None, 'score': None, 'similarity': 0.0, 'ingredients': [],
            'status': 'no_candidate'}


# ------------------------------------------------------------ brand table + resolver
def load_brand_map(path):
    """CSV columns: brand, ingredients (separated by ';'), origin, verified. -> {cleaned brand: [canonical ingredients]}."""
    m = {}
    if path and os.path.exists(path):
        df = pd.read_csv(path)
        for b, ings in zip(df['brand'], df['ingredients']):
            key, lst = _clean_query(str(b)), [canon(x) for x in str(ings).split(';') if x.strip()]
            if key and lst:
                m[key] = lst
    return m


def resolve_drug(text, brand_map, cache, min_similarity=85, brand_fuzzy=90):
    """Free text -> ingredients, trying (1) the brand table, (2) a typo-tolerant brand-table match, (3) RxNorm.
    status 'ok' = trusted; anything else must be treated as UNRECOGNISED (never guessed)."""
    q = _clean_query(text) or text.lower().strip()
    if q in brand_map:
        return {'query': text, 'status': 'ok', 'route': 'brand_table', 'matched': q, 'similarity': 100.0, 'ingredients': brand_map[q]}
    if brand_map:
        near = max(brand_map, key=lambda k: name_similarity(q, k))
        sim = name_similarity(q, near)
        if sim >= brand_fuzzy:
            return {'query': text, 'status': 'ok', 'route': 'brand_table_fuzzy', 'matched': near, 'similarity': round(sim, 1), 'ingredients': brand_map[near]}
    r = rx_normalize(text, cache, min_similarity=min_similarity)
    r['route'] = 'rxnorm:' + str(r.get('route'))
    return r


# ------------------------------------------------- label sentences about cardiac topics
# NOTE on tiers: they describe WHERE in the label a statement sits (boxed/contraindication/warning = strong, adverse reaction or
# interaction = moderate, overdose/pharmacology = weak). They are label prominence, NOT clinical strength or causal proof.
SECTION_GROUP = {   # openFDA label field -> (group, strength)
    'boxed_warning': ('boxed', 3), 'contraindications': ('contraindication', 3),
    'warnings': ('warning', 3), 'warnings_and_cautions': ('warning', 3), 'warnings_and_precautions': ('warning', 3),
    'precautions': ('warning', 3), 'general_precautions': ('warning', 3),
    'adverse_reactions': ('adverse', 2), 'drug_interactions': ('interaction', 2),
    'overdosage': ('overdose', 1), 'clinical_pharmacology': ('pharmacology', 1),
    'pharmacodynamics': ('pharmacology', 1), 'mechanism_of_action': ('pharmacology', 1),
}
TOPICS = {
    'qt': re.compile(r'\bQ[-‐-–]?T(?:c|cF|cB|cI)?\b|[Tt]orsades?'),
    'bradycardia': re.compile(r'[Bb]radycardi|[Bb]radyarrhythmi|heart rate (?:decreas|reduc|slow)|slow(?:ing|ed|s)? (?:of )?(?:the )?heart rate|sinus (?:pause|arrest)'),
    'av_block': re.compile(r'(?:[Aa]trioventricular|\bA-?V\b) (?:block|conduction)|[Hh]eart block|\bPR (?:interval|prolong)'),
}
_W = r'(?:\w+\s+)'
NEG = [re.compile(p, re.I) for p in (
    rf'\b(?:no|without)\s+{_W}{{0,3}}?(?:effects?|changes?|increases?|prolongation|differences?|impact)\b[^.]{{0,40}}\b(?:QT|Q-T|torsade|bradycardi|heart block|AV|PR interval)',
    rf'\b(?:did|does|do|was|were|has|have|had|is|are)\s+not\s+{_W}{{0,2}}?(?:prolong\w*|affect\w*|alter\w*|increase\w*|cause\w*|produce\w*|induce\w*|lengthen\w*|associated|shown|show|demonstrate\w*|result\w*|appear\w*|observed|reported|seen|noted)',
    rf'\bnot\s+(?:been\s+)?(?:shown|found|observed|reported|associated)\s+to\s+{_W}{{0,2}}?(?:prolong|affect|alter|increase|cause)',
    rf'\bno\s+(?:evidence|signs?|cases?|reports?)\s+of\s+[^.]{{0,40}}\b(?:QT|Q-T|torsade|bradycardi|heart block)',
    r'\bQ-?T[^.]{0,60}\b(?:was|were)\s+(?:not|unchanged|similar to placebo)',
)]
POS = re.compile(r'\bprolong\w*|\blengthen\w*|[Tt]orsade|\b(?:increas\w*)\b[^.]{0,60}\bQ-?T|\bQ-?T[^.]{0,60}\b(?:increas\w*|prolong\w*)|'
                 r'\brisk\b|\bcaus\w*|\binduc\w*|\bproduc\w*|\breport\w*|\bobserv\w*|\bsevere|\bsymptomatic|\barrhythmi', re.I)

def split_sentences(text):
    text = re.sub(r'\s+', ' ', text)
    return [s.strip() for s in re.split(r'(?<=[.!?;])\s+(?=[A-Z0-9\[(•])', text) if s.strip()]


def classify_sentence(sentence, topic, group):
    """-> 'negated' | 'positive' | 'mention' | None (topic not mentioned)."""
    if not TOPICS[topic].search(sentence):
        return None
    if any(p.search(sentence) for p in NEG):
        return 'negated'
    if group in ('boxed', 'contraindication', 'warning', 'adverse') or POS.search(sentence):
        return 'positive'
    return 'mention'


def extract_label_evidence(label, max_per_topic=15, max_len=420):
    """One openFDA label dict -> {topic: [ {group, strength, field, text, cls} ]} (only sentences about the topic).
    Roles are NOT stored: they are computed from the text at summary time, so the role rules can change without re-fetching."""
    out = {t: [] for t in TOPICS}
    for field, (group, strength) in SECTION_GROUP.items():
        v = label.get(field)
        if not v:
            continue
        text = ' '.join(v) if isinstance(v, list) else str(v)
        for s in split_sentences(text):
            for t in TOPICS:
                cls = classify_sentence(s, t, group)
                if cls and len(out[t]) < max_per_topic:
                    out[t].append({'group': group, 'strength': strength, 'field': field, 'cls': cls, 'text': s[:max_len]})
    return out


def label_meta(label):
    o = label.get('openfda') or {}
    gen = o.get('generic_name') or o.get('substance_name') or []
    low = ' '.join(gen).lower()
    single = len(gen) == 1 and not re.search(r'\band\b|,|/|\bwith\b|\+', low)
    return {'set_id': label.get('set_id'), 'id': label.get('id'), 'effective_time': label.get('effective_time'),
            'generic_name': gen[:3], 'brand_name': (o.get('brand_name') or [])[:3], 'single_ingredient': bool(single)}


CACHE_VERSION = 'fda2'      # bump when extraction rules change (the cache stores extracted evidence)


def fda_fetch(ingredient, cache, api_key=None, limit=10):
    """Retrieve up to `limit` recent labels for an ingredient and cache the extracted evidence. None = network failure."""
    key = f'{CACHE_VERSION}|{ingredient.lower().strip()}|{limit}'
    if cache.has(key):
        return cache.get(key)
    base = {'api_key': api_key} if api_key else {}
    results, failed = None, False
    for field in ('openfda.generic_name', 'openfda.substance_name'):
        params = {**base, 'search': f'{field}:"{ingredient}"', 'limit': limit, 'sort': 'effective_time:desc'}
        js = http_get(FDA_LABEL, params)
        if js and js.get('_status') == 400:           # some deployments reject the sort field
            params.pop('sort')
            js = http_get(FDA_LABEL, params)
        if js is None:
            failed = True; break
        if js.get('results'):
            results = js['results']; break
    if failed:
        return None
    rec = {'ingredient': ingredient, 'labels': []}
    for lab in results or []:
        m = label_meta(lab)
        m['evidence'] = extract_label_evidence(lab)
        rec['labels'].append(m)
    cache.set(key, rec)
    return rec


# ------------------------------------------------------------- roles of a label sentence
ROLES = ('drug_effect', 'patient_condition', 'interaction')
_STOPW = r'(?:are|is|were|was|may|can|could|should|will|would|risk of|have been|has been|at (?:a )?(?:higher|increased|greater) risk)'
_ACTOR = r'(?:patients?|subjects?|individuals?|persons?|people|those|women|men|children|elderly|adults)'
_WITH = r'(?:with|who have|who has|having|had|suffering from|exhibiting|showing|experiencing)'
_INTERACTION = re.compile(r'\b(?:concomitant\w*|co-?administ\w*|coadminist\w*|in combination with|combined with|taken with|used with|'
                          r'given with|administered with|other drugs|drugs that|medications that|agents that|inhibitors? of|CYP\w+)\b', re.I)
_ROLE_RX = {}


def _gap(n, stop=_STOPW):
    """Up to n characters inside one sentence that do not cross a verb like 'are', 'may', 'risk of'."""
    return r'(?:(?!\b' + stop + r'\b)[^.]){0,' + str(n) + r'}?'


def _role_rx(topic):
    if topic not in _ROLE_RX:
        T = '(?:' + TOPICS[topic].pattern + ')'
        flags = re.I
        patient = [
            re.compile(r'\b' + _ACTOR + r'\b' + _gap(120) + r'\b' + _WITH + r'\b' + _gap(90) + T, flags),
            re.compile(r'\b(?:history of|known|congenital|pre-?existing|underlying|uncorrected|clinically significant|acquired|baseline)\b' + _gap(60) + T, flags),
            re.compile(r'\b(?:contraindicated|should not be (?:used|given|administered|initiated)|do not (?:use|administer|give|initiate)|avoid(?:ed)?(?: use)?|'
                       r'not recommended|must not)\b' + _gap(100, r'(?:were|was|can|could|will|would|have been|has been)') + T, flags),
            re.compile(T + _gap(40) + r'\b(?:risk factors?|predispos\w*|susceptib\w*)', flags)]
        effect = [
            re.compile(r'\b(?:caus\w*|induc\w*|produc\w*|result\w* in|lead\w* to|associated with|increas\w*|risk of|prolongs|'
                       r'(?:may|can|could|will|might|to)\s+(?:also\s+)?prolong)\b' + _gap(60, r'(?:with patients|in patients)') + T, flags),
            re.compile(T + _gap(50, r'(?:in patients|patients)') + r'\b(?:have been|has been|reported|observed|occur\w*|was associated|were associated)\b', flags)]
        _ROLE_RX[topic] = {'patient': patient, 'effect': effect}
    return _ROLE_RX[topic]


def sentence_roles(sentence, topic, group):
    """Which relation does this label sentence express?
    drug_effect: the drug may cause / is reported with the finding.
    patient_condition: caution or contraindication in patients who already have the finding (or its risk factors).
    interaction: the statement is about co-administration with other drugs.
    A sentence can carry several roles. Rule-based; the sentence is always shown next to its role so it can be checked."""
    if group == 'interaction':
        return ['interaction']
    rx = _role_rx(topic)
    hits = [bool(p.search(sentence)) for p in rx['patient']]
    if _INTERACTION.search(sentence):
        # a sentence about co-medication is an interaction statement, not the drug's own effect; it only also counts as
        # patient_condition when it explicitly says 'patients with <finding>'
        return ['interaction', 'patient_condition'] if hits[0] else ['interaction']
    roles = []
    if any(hits):
        roles.append('patient_condition')
    if any(p.search(sentence) for p in rx['effect']):
        roles.append('drug_effect')
    if not roles:
        roles = ['patient_condition'] if group == 'contraindication' else ['drug_effect']
    return roles


# ------------------------------------------------------------- evidence tiers
TIER_RANK = {'no_label': -1, 'none': 0, 'negated_only': 1, 'mention': 2, 'weak': 3, 'moderate': 4, 'strong': 5}
_STRENGTH_TIER = {3: 'strong', 2: 'moderate', 1: 'weak'}


def _label_tier(ev):
    pos = [e for e in ev if e['cls'] == 'positive']
    if pos:
        return _STRENGTH_TIER[max(e['strength'] for e in pos)]
    if any(e['cls'] == 'mention' for e in ev):
        return 'mention'
    if any(e['cls'] == 'negated' for e in ev):
        return 'negated_only'
    return 'none'


def _cite(l, e, roles):
    return {**e, 'roles': roles, 'set_id': l['set_id'], 'effective_time': l['effective_time'], 'brand': l['brand_name'][:1],
            'url': f"https://dailymed.nlm.nih.gov/dailymed/lookup.cfm?setid={l['set_id']}" if l['set_id'] else None}


def summarize_evidence(rec, topic, max_labels=5, top_k=3):
    """Aggregate over labels (single-ingredient labels preferred), overall and per role.
    'none' = no statement found in the retrieved labels (NOT proof of safety); 'no_label' = nothing retrieved
    (-> insufficient evidence). n_positive = number of the used labels that contain a positive statement."""
    if not rec or not rec.get('labels'):
        empty_roles = {r: {'tier': 'no_label', 'n_positive': 0, 'evidence': []} for r in ROLES}
        return {'topic': topic, 'tier': 'no_label', 'n_labels': 0, 'n_positive': 0, 'combination_only': False, 'evidence': [], 'by_role': empty_roles}
    labs = [l for l in rec['labels'] if l['single_ingredient']]
    combo_only = not labs
    labs = (labs or rec['labels'])[:max_labels]
    tiers = [_label_tier(l['evidence'][topic]) for l in labs]
    tier = max(tiers, key=lambda t: TIER_RANK[t])
    ev, per_role = [], {r: {'labels': 0, 'strength': 0, 'ev': []} for r in ROLES}
    for l in labs:
        seen_roles = set()
        for e in l['evidence'][topic]:
            if e['cls'] != 'positive':
                continue
            roles = sentence_roles(e['text'], topic, e['group'])
            c = _cite(l, e, roles)
            ev.append(c)
            for r in roles:
                per_role[r]['ev'].append(c)
                per_role[r]['strength'] = max(per_role[r]['strength'], e['strength'])
                seen_roles.add(r)
        for r in seen_roles:
            per_role[r]['labels'] += 1
    ev.sort(key=lambda e: -e['strength'])
    by_role = {}
    for r in ROLES:
        p = per_role[r]
        p['ev'].sort(key=lambda e: -e['strength'])
        by_role[r] = {'tier': _STRENGTH_TIER.get(p['strength'], 'none'), 'n_positive': p['labels'], 'evidence': p['ev'][:2]}
    return {'topic': topic, 'tier': tier, 'n_labels': len(labs), 'combination_only': combo_only,
            'n_positive': sum(t in ('strong', 'moderate', 'weak') for t in tiers), 'evidence': ev[:top_k], 'by_role': by_role}


# -------------------------------------------------------------------- DDInter
LEVEL_RANK = {'major': 3, 'moderate': 2, 'minor': 1, 'unknown': 0}


def _find(cols, pats):
    for p in pats:
        for c in cols:
            if re.fullmatch(p, c.lower().strip()):
                return c
    return None


class DDI:
    """Pairwise interaction index from DDInter CSVs (any number of files; column names are auto-detected)."""

    def __init__(self, folder):
        self.pairs, self.files, self.columns = {}, [], {}
        paths = sorted(glob.glob(os.path.join(folder, '*.csv'))) if folder and os.path.isdir(folder) else []
        for f in paths:
            df = pd.read_csv(f)
            cols = list(df.columns)
            a = _find(cols, [r'drug_?a(?:_name)?', r'drug[ _]?1', r'drug_?a_?name'])
            b = _find(cols, [r'drug_?b(?:_name)?', r'drug[ _]?2', r'drug_?b_?name'])
            lv = _find(cols, [r'level', r'severity', r'interaction_?level'])
            if not (a and b):
                name_like = [c for c in cols if 'drug' in c.lower() and 'id' not in c.lower()]
                if len(name_like) >= 2:
                    a, b = name_like[:2]
            self.columns[os.path.basename(f)] = {'drug_a': a, 'drug_b': b, 'level': lv, 'all': cols[:12]}
            if not (a and b):
                continue
            self.files.append(os.path.basename(f))
            for x, y, l in zip(df[a], df[b], df[lv] if lv else ['unknown'] * len(df)):
                k = frozenset((canon(x), canon(y)))
                l = str(l).strip().lower()
                l = l if l in LEVEL_RANK else 'unknown'
                if k not in self.pairs or LEVEL_RANK[l] > LEVEL_RANK[self.pairs[k]]:
                    self.pairs[k] = l
        self.drugs = sorted({d for k in self.pairs for d in k})

    def __len__(self): return len(self.pairs)

    def level(self, a, b):
        return self.pairs.get(frozenset((canon(a), canon(b))))

    def check(self, ingredients):
        """All interacting pairs among a list of ingredient names, strongest first."""
        ings = sorted({canon(i) for i in ingredients})
        out = []
        for i in range(len(ings)):
            for j in range(i + 1, len(ings)):
                l = self.pairs.get(frozenset((ings[i], ings[j])))
                if l:
                    out.append({'drug_a': ings[i], 'drug_b': ings[j], 'level': l, 'source': 'DDInter'})
        return sorted(out, key=lambda r: -LEVEL_RANK[r['level']])


# ---------------------------------------------------------- CredibleMeds (optional)
def cm_category(text):
    t = str(text).lower()
    if 'known' in t or t.strip() == 'kr': return 'known'
    if 'possible' in t or t.strip() == 'pr': return 'possible'
    if 'conditional' in t or t.strip() == 'cr': return 'conditional'
    if 'congenital' in t or 'avoid' in t or 'clqt' in t: return 'congenital_avoid'
    return 'other'


def load_crediblemeds(folder):
    """User-supplied CredibleMeds export (csv/xlsx). Returns ({canon_name: category}, info). Used locally only."""
    paths = []
    if folder and os.path.isdir(folder):
        for ext in ('*.csv', '*.xlsx', '*.xls'):
            paths += glob.glob(os.path.join(folder, ext))
    info, mapping = {'files': [os.path.basename(p) for p in paths]}, {}
    for p in paths:
        df = pd.read_csv(p) if p.endswith('.csv') else pd.read_excel(p)
        cols = list(df.columns)
        nc = _find(cols, [r'.*generic.*', r'.*drug.*name.*', r'.*name.*', r'.*drug.*'])
        cc = _find(cols, [r'.*categor.*', r'.*risk.*', r'.*group.*'])
        info[os.path.basename(p)] = {'name_col': nc, 'category_col': cc, 'columns': cols[:10]}
        if nc and cc:
            for n, c in zip(df[nc], df[cc]):
                if isinstance(n, str):
                    mapping[canon(n)] = cm_category(c)
    return mapping, info


In [ ]:
import importlib, json, datetime, numpy as np, pandas as pd
from tqdm.auto import tqdm
import cardiosafe_kb as K
importlib.reload(K)
API_KEY = os.environ.get('OPENFDA_API_KEY')
try:
    from google.colab import userdata
    API_KEY = userdata.get('OPENFDA_API_KEY') or API_KEY
except Exception:
    pass
rx_cache = K.JsonCache(KB + '/cache_rxnorm.json'); fda_cache = K.JsonCache(KB + '/cache_openfda.json')
print('openFDA key:', 'found' if API_KEY else 'not set', '| cache version:', K.CACHE_VERSION)

## 1. Label evidence by role (re-fetched with the new extraction rules)
Each cell reads `tier n/N`: strongest label section in which this role appears, and in how many (n) of the N labels used. Sections: boxed warning / contraindication / warning = strong, adverse reaction / interaction = moderate, overdose / pharmacology = weak. **Tiers show label prominence, not clinical strength.**

In [ ]:
POS = {'qt': ['azithromycin', 'clarithromycin', 'erythromycin', 'levofloxacin', 'moxifloxacin', 'ciprofloxacin', 'ondansetron', 'haloperidol',
              'citalopram', 'escitalopram', 'methadone', 'amiodarone', 'sotalol', 'hydroxychloroquine', 'fluconazole', 'quetiapine', 'donepezil', 'chlorpromazine'],
       'bradycardia': ['metoprolol', 'atenolol', 'propranolol', 'digoxin', 'verapamil', 'diltiazem', 'amiodarone', 'donepezil'],
       'av_block': ['metoprolol', 'atenolol', 'propranolol', 'digoxin', 'verapamil', 'diltiazem', 'amiodarone']}
NEG = ['metformin', 'atorvastatin', 'lisinopril', 'amlodipine', 'levothyroxine', 'acetaminophen', 'simvastatin', 'gabapentin']
SANITY = sorted(set(sum(POS.values(), []) + NEG))
FLAGGED = ('strong', 'moderate', 'weak')
records = {d: K.fda_fetch(d, fda_cache, API_KEY) for d in tqdm(SANITY)}
fda_cache.save()
print('requests this session:', K.STATS, '| drugs without a result:', [d for d, r in records.items() if r is None])

def cell(s, role):
    r = s['by_role'][role]
    return '-' if r['tier'] in ('none', 'no_label') else f"{r['tier']} {r['n_positive']}/{s['n_labels']}"
for t in K.TOPICS:
    rows = []
    for d in SANITY:
        s = K.summarize_evidence(records[d], t)
        rows.append({'drug': d, 'drug_effect': cell(s, 'drug_effect'), 'patient_condition': cell(s, 'patient_condition'), 'interaction': cell(s, 'interaction'), 'overall': s['tier']})
    print(f'\n=== topic: {t} ===')
    print(pd.DataFrame(rows).set_index('drug').to_string())

In [ ]:
# Detection re-scored by ROLE. A drug 'has the effect' only if a drug_effect sentence exists.
rows = []
for t in K.TOPICS:
    pos = [d for d in POS[t] if records[d]]; neg = [d for d in NEG if records[d]]
    eff = lambda d: K.summarize_evidence(records[d], t)['by_role']['drug_effect']['tier'] in FLAGGED
    anyr = lambda d: K.summarize_evidence(records[d], t)['tier'] in FLAGGED
    others = [d for d in SANITY if d not in POS[t] and d not in NEG and records[d]]
    rows.append({'topic': t, 'expected_pos': len(pos), 'effect_flagged': sum(eff(d) for d in pos), 'any_role_flagged': sum(anyr(d) for d in pos),
                 'effect_missed': ', '.join(d for d in pos if not eff(d)), 'neg_effect_flags': ', '.join(d for d in neg if eff(d)),
                 'neg_any_role_flags': ', '.join(d for d in neg if anyr(d)),
                 'other_drugs_with_effect_statement': ', '.join(d for d in others if eff(d))})
print(pd.DataFrame(rows).set_index('topic').to_string())

### The actual sentences, per role
These are the cases from your diagnostics. Check whether each printed role is right for the printed sentence. Where it is wrong, tell me the drug, topic and role; those are the rule fixes.

In [ ]:
CASES = [('azithromycin', 'bradycardia'), ('ciprofloxacin', 'bradycardia'), ('citalopram', 'bradycardia'), ('amlodipine', 'bradycardia'),
         ('ondansetron', 'bradycardia'), ('digoxin', 'qt'), ('escitalopram', 'qt'), ('donepezil', 'qt'), ('erythromycin', 'qt'),
         ('haloperidol', 'av_block'), ('hydroxychloroquine', 'av_block'), ('atenolol', 'av_block'), ('sotalol', 'qt'), ('metoprolol', 'bradycardia')]
for drug, topic in CASES:
    s = K.summarize_evidence(records[drug], topic)
    print(f'\n=== {drug.upper()} / {topic}  (labels used {s["n_labels"]}{", combination labels only" if s["combination_only"] else ""})')
    shown = False
    for role in K.ROLES:
        r = s['by_role'][role]
        if r['evidence']:
            e = r['evidence'][0]; shown = True
            print(f'  {role:18s} {r["tier"]:8s} {r["n_positive"]}/{s["n_labels"]} | [{e["group"]}] {e["text"][:230]}')
    if not shown: print('  (no positive statements found)')

### Chlorpromazine
Notebook 07b found no QT sentence in any field of its 10 labels, so the evidence tier for chlorpromazine is `none`. For the report this is the honest wording: **"no QT statement found in the FDA labels retrieved"**, which is *not* the same as "no QT risk". The CredibleMeds comparison (section 4), once you provide the list, tells us how often this happens.

## 2. Drug-name resolution
`kb/brand_map.csv` is a **starter table that I wrote from general knowledge** (US brands plus common Indian brands). Every row has `verified = no`. Please check it and add the brands your own prescriptions use; the notebook never overwrites your edits. Combination brands (for example *Ecosprin AV*, *Pantocid DSR*) are deliberately absent: they resolve as unrecognised until someone adds them, which is the safe outcome.

In [ ]:
BRANDS = [('Zithromax', 'azithromycin', 'US'), ('Zofran', 'ondansetron', 'US'), ('Augmentin', 'amoxicillin;clavulanate', 'US'), ('Biaxin', 'clarithromycin', 'US'), ('Cipro', 'ciprofloxacin', 'US'), ('Levaquin', 'levofloxacin', 'US'), ('Avelox', 'moxifloxacin', 'US'), ('Diflucan', 'fluconazole', 'US'), ('Celexa', 'citalopram', 'US'), ('Lexapro', 'escitalopram', 'US'), ('Seroquel', 'quetiapine', 'US'), ('Haldol', 'haloperidol', 'US'), ('Thorazine', 'chlorpromazine', 'US'), ('Aricept', 'donepezil', 'US'), ('Plaquenil', 'hydroxychloroquine', 'US'), ('Cordarone', 'amiodarone', 'US'), ('Pacerone', 'amiodarone', 'US'), ('Betapace', 'sotalol', 'US'), ('Dolophine', 'methadone', 'US'), ('Lopressor', 'metoprolol', 'US'), ('Toprol', 'metoprolol', 'US'), ('Tenormin', 'atenolol', 'US'), ('Inderal', 'propranolol', 'US'), ('Lanoxin', 'digoxin', 'US/IN'), ('Cardizem', 'diltiazem', 'US'), ('Calan', 'verapamil', 'US'), ('Norvasc', 'amlodipine', 'US'), ('Lipitor', 'atorvastatin', 'US'), ('Zocor', 'simvastatin', 'US'), ('Crestor', 'rosuvastatin', 'US'), ('Glucophage', 'metformin', 'US'), ('Zestril', 'lisinopril', 'US'), ('Prinivil', 'lisinopril', 'US'), ('Neurontin', 'gabapentin', 'US'), ('Synthroid', 'levothyroxine', 'US'), ('Tylenol', 'acetaminophen', 'US'), ('Advil', 'ibuprofen', 'US'), ('Motrin', 'ibuprofen', 'US'), ('Dolo', 'paracetamol', 'IN'), ('Crocin', 'paracetamol', 'IN'), ('Calpol', 'paracetamol', 'IN/UK'), ('Azithral', 'azithromycin', 'IN'), ('Azee', 'azithromycin', 'IN'), ('Ecosprin', 'aspirin', 'IN'), ('Storvas', 'atorvastatin', 'IN'), ('Atorva', 'atorvastatin', 'IN'), ('Telma', 'telmisartan', 'IN'), ('Amlong', 'amlodipine', 'IN'), ('Metolar', 'metoprolol', 'IN'), ('Cardace', 'ramipril', 'IN'), ('Ciplox', 'ciprofloxacin', 'IN'), ('Cifran', 'ciprofloxacin', 'IN'), ('Ondem', 'ondansetron', 'IN'), ('Emeset', 'ondansetron', 'IN'), ('Pantocid', 'pantoprazole', 'IN'), ('Omez', 'omeprazole', 'IN'), ('Glycomet', 'metformin', 'IN'), ('Clavam', 'amoxicillin;clavulanate', 'IN'), ('Nexito', 'escitalopram', 'IN'), ('HCQS', 'hydroxychloroquine', 'IN'), ('Eltroxin', 'levothyroxine', 'IN/UK'), ('Thyronorm', 'levothyroxine', 'IN'), ('Aten', 'atenolol', 'IN'), ('Largactil', 'chlorpromazine', 'IN/UK'), ('Betaloc', 'metoprolol', 'IN/UK'), ('Dilzem', 'diltiazem', 'IN'), ('Concor', 'bisoprolol', 'IN/US'), ('Rosuvas', 'rosuvastatin', 'IN')]
bm_path = KB + '/brand_map.csv'
if not os.path.exists(bm_path):
    pd.DataFrame(BRANDS, columns=['brand', 'ingredients', 'origin']).assign(verified='no').to_csv(bm_path, index=False)
    print('wrote starter brand table:', bm_path)
bm = K.load_brand_map(bm_path)
print(len(bm), 'brands loaded')

tests = ['azithromycin', 'Zithromax', 'Zofran', 'Zofran 4 mg tab', 'ondansetron 4 mg', 'ondansetrone', 'metoprolol succinate 50 mg', 'Augmentin 625',
         'Azithral 500', 'Dolo 650', 'Crocin', 'paracetamol 500 mg', 'Ecosprin 75', 'Ecosprin AV 75', 'Ecospirin 75', 'Telma 40', 'Pantocid DSR', 'Glycomet GT',
         'amiodarone 200 mg', 'Xyzzyflam 10 mg']
rows = []
for t in tests:
    r = K.resolve_drug(t, bm, rx_cache)
    rows.append({'input': t, 'status': r['status'], 'route': r['route'], 'similarity': r.get('similarity'), 'matched': r.get('matched'), 'ingredients': '; '.join(r['ingredients'])})
rx_cache.save()
print(pd.DataFrame(rows).to_string(index=False))
print('\nroutes used:', pd.Series([r['route'] for r in rows]).value_counts().to_dict())
print('Expected: Ecosprin AV, Pantocid DSR, Glycomet GT (combination brands) and Xyzzyflam should NOT be status ok.')

## 3. DDInter interactions and CredibleMeds (unchanged; skipped automatically if you have not added the files yet)

In [ ]:
ddi = K.DDI(KB + '/ddinter')
if len(ddi) == 0:
    print('No DDInter data loaded yet. Put the downloaded CSV files in', KB + '/ddinter', 'and re-run this cell.')
else:
    print('files:', ddi.files)
    for f, c in ddi.columns.items(): print(' ', f, '->', {k: v for k, v in c.items() if k != 'all'})
    print(f'{len(ddi):,} unique drug pairs | {len(ddi.drugs):,} distinct drug names')
    print(pd.Series(list(ddi.pairs.values())).value_counts().to_dict())
    inn = [d for d in SANITY if K.canon(d) in set(ddi.drugs)]
    print(f'sanity drugs present in DDInter: {len(inn)}/{len(SANITY)} | missing: {sorted(set(SANITY) - set(inn))}')
    print(pd.DataFrame(ddi.check(POS['qt'])).head(15).to_string(index=False))

In [ ]:
cm, cm_info = K.load_crediblemeds(KB + '/crediblemeds')
if not cm:
    print('CredibleMeds list not provided (or columns not detected). Detected:', cm_info)
else:
    print('detected columns:', {k: v for k, v in cm_info.items() if k != 'files'}, '| categories:', pd.Series(list(cm.values())).value_counts().to_dict())
    rng = np.random.default_rng(0)
    risky = [d for d, c in cm.items() if c in ('known', 'possible', 'conditional') and ' ' not in d and '+' not in d]
    sample_pos = list(rng.choice(risky, min(120, len(risky)), replace=False))
    pool_neg = [d for d in (ddi.drugs if len(ddi) else NEG) if d not in cm and ' ' not in d and '+' not in d]
    sample_neg = list(rng.choice(pool_neg, min(60, len(pool_neg)), replace=False))
    out = []
    for d in tqdm(sample_pos + sample_neg):
        rec = K.fda_fetch(d, fda_cache, API_KEY)
        if rec is None: continue
        s = K.summarize_evidence(rec, 'qt')
        out.append({'drug': d, 'cm': cm.get(d, 'not_listed'), 'effect_tier': s['by_role']['drug_effect']['tier'], 'overall_tier': s['tier']})
    fda_cache.save()
    r = pd.DataFrame(out); r['effect_flagged'] = r.effect_tier.isin(FLAGGED); r['any_flagged'] = r.overall_tier.isin(FLAGGED)
    tab = r.groupby('cm').agg(n=('drug', 'count'), effect_flagged=('effect_flagged', 'sum'), any_role_flagged=('any_flagged', 'sum'),
                              no_label=('overall_tier', lambda s: int((s == 'no_label').sum())))
    tab['effect_rate'] = tab.effect_flagged / tab.n; tab['any_role_rate'] = tab.any_role_flagged / tab.n
    print(tab.round(3).to_string()); tab.to_csv(KB + '/results_07c_crediblemeds_agreement.csv')
    print('Known-risk drugs without any drug_effect statement in the labels:', r[(r.cm == 'known') & ~r.effect_flagged].drug.tolist())

## 4. Export

In [ ]:
rx_cache.save(); fda_cache.save()
kb_json = {'retrieved': datetime.date.today().isoformat(), 'cache_version': K.CACHE_VERSION, 'drugs': {}}
for d, rec in records.items():
    if rec: kb_json['drugs'][d] = {t: K.summarize_evidence(rec, t) for t in K.TOPICS}
json.dump(kb_json, open(KB + '/kb_sanity.json', 'w'), indent=1)
json.dump({'retrieved': kb_json['retrieved'], 'cache_version': K.CACHE_VERSION, 'flag_tiers': list(FLAGGED), 'roles': list(K.ROLES),
           'brand_table_rows': len(bm), 'ddinter_pairs': len(ddi), 'crediblemeds_provided': bool(cm)}, open(KB + '/kb_config.json', 'w'), indent=1)
print('saved kb_sanity.json and kb_config.json in', KB)

## Next
Send the raw text of the role tables, the detection table, the sentence list and the resolver table. Then I fix any wrong roles and move on to notebook 08 (the prescription pipeline: synthetic prescriptions, OCR comparison, drug extraction), which will use this resolver.